# Bonds

Author: Sebastien Gurrieri, sebgur@gmail.com

Bond definitions, with calculation of typical quantities (e.g. yields, spreads, durations). 

In [ ]:
import datetime as dt
import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
import sdevpy as sd
from sdevpy import pricingcontext
# from sdevpy.utilities import dates as dts
from sdevpy.utilities.scalendar import BDC, make_calendar
from sdevpy.utilities.sdaycount import DayCount
from sdevpy.utilities import schedules as sch
# from sdevpy.utilities import timegrids
from sdevpy.rates import bonds, swaps
from sdevpy.calibration.rates import yieldcurve as ycrv


print("SDevPy version: " + sd.__version__)
ctx = pricingcontext.default_pricing_context()
date_fmt = '{:%d-%b-%Y}'

SDevPy version: 1.2


### Static Definition
Definition of the bond from static inputs (dates, conventions, etc.). Display of the cash-flow schedule and accruals.

In [ ]:
# Bond definition
issue = dt.date(2025, 1, 10)
maturity = dt.date(2036, 2, 15)
coupon = 0.04
conv = sch.LegConvention(frequency='6M', daycount=DayCount.ACTACT_ICMA, calendar='USD', bdc=BDC.F,
                         adjust_accrual=False, eom=True)

# Calculate schedule
scheduler = sch.make_bond_scheduler(issue, maturity, conv)
cpn_schedule = scheduler.coupon_schedule(coupon)

# View schedule
schedule_df = pd.DataFrame(cpn_schedule)
print(f"Bond {coupon:.2%} {maturity:%d-%b-%Y}: {scheduler.schedule.n_periods} coupons,"
      f" {scheduler.coupons_per_year} per year, redemption on {scheduler.maturity}")
schedule_df.style.format({'Accrual start': date_fmt, 'Accrual end': date_fmt, 'Payment': date_fmt,
                          'Accrual fraction': '{:.6f}', 'Coupon': '{:.6f}'})

Bond 4.00% 15-Feb-2036: 23 coupons, 2 per year, redemption on 2036-02-15


,Accrual start,Accrual end,Payment,Days,Accrual fraction,Coupon
0,10-Jan-2025,15-Feb-2025,18-Feb-2025,36,0.097826,0.391304
1,15-Feb-2025,15-Aug-2025,15-Aug-2025,181,0.500000,2.000000
2,15-Aug-2025,15-Feb-2026,17-Feb-2026,184,0.500000,2.000000
3,15-Feb-2026,15-Aug-2026,17-Aug-2026,181,0.500000,2.000000
4,15-Aug-2026,15-Feb-2027,16-Feb-2027,184,0.500000,2.000000
5,15-Feb-2027,15-Aug-2027,16-Aug-2027,181,0.500000,2.000000
6,15-Aug-2027,15-Feb-2028,15-Feb-2028,184,0.500000,2.000000
7,15-Feb-2028,15-Aug-2028,15-Aug-2028,182,0.500000,2.000000
8,15-Aug-2028,15-Feb-2029,15-Feb-2029,184,0.500000,2.000000
9,15-Feb-2029,15-Aug-2029,15-Aug-2029,181,0.500000,2.000000


### Basic Quantities
Now we assume the bond price is given on a certain valuation date and we calculate a number of relevant quantities for the bond.

** Definitions **
* Yield (to maturity): single rate that discounts all the cash flows back to the dirty price, compounded semi-annually (the market quote).
* Yield (annual): the same yield with annual compounding, comparable with annually compounded yields.
* Yield (current): annual coupon divided by clean price.

* Macaulay duration: average time to the cash flows, weighted by their present value.
* Modified duration: percentage price change for a 1-unit change in yield. A modified duration of 7.63 means the price moves about 7.63% for 100bp.
* DV01: same risk in price terms. Here it's 0.0757 per 100 notional per bp, so $757 per bp on $10m.
* Convexity: how much the duration changes as yield moves (i.e. curvature of the price/yield curve).

* Z-spread: constant spread added to every zero rate of the curve that reprices the bond to its market price.
* I-spread: bond's yield minus the swap rate interpolated at the bond's maturity.

TODO:
* Fitting many bonds at once: if we fit many bonds together, the flatten + np.add.reduceat draft in claude/scratch/schedules.py is the way to vectorise. That draft predates later design changes, so it needs updating.* Spread duration and key-rate durations: sensitivity to the Z-spread, and to individual curve points. They come naturally once you have curve construction and bumping.
* Build the treasury curve
* Final coupon period: a Treasury in its last coupon period uses a simple-yield convention, which isn't handled. It doesn't affect fitting to dirty prices, but it does if we fit to quoted yields.
* G-spread: yield minus the government curve's yield at the same maturity. Needs a Treasury curve.
* Asset swap spread (ASW): spread over the floating leg of a swap that turns the bond into a floating-rate asset. It's another common YAS number, and it needs a floating-leg annuity on the swap curve. It would be a small addition on top of swaps.py.
* Settlement: there's no settlement-date helper yet. The notebook uses add_business_days(valdate, 1). T+1 is right for USTs, but a proper helper would be cleaner before fitting many bonds.

In [ ]:
# Market inputs
valdate = dt.datetime(2025, 12, 15)
clean_price = 98.50
settlement = make_calendar(conv.calendar).add_business_days(valdate.date(), 1) # UST: T+1

# Retrieve swap curve
dataset = ctx.calib_repo[valdate]
rfr_curve = dataset.get_yieldcurve("USD.SOFR.1D").discount_float

# Price, accrued, yield
bf = bonds.bond_flows(scheduler, settlement, valdate)
accrued = bonds.accrued_interest(bf, coupon)
dirty = clean_price + accrued
ytm = bonds.yield_from_dirty(bf, coupon, dirty)
starts, ends = scheduler.accrual_dates()
i = int(np.searchsorted(ends, np.datetime64(settlement, 'D'), side='right'))

# Spreads: Z-spread over the curve, I-spread over the par swap rate to the bond's maturity
zspread = bonds.z_spread(bf, coupon, dirty, rfr_curve)
sofr = sch.LegConvention('1Y', DayCount.ACT360, 'USD', pay_lag=2)
usd_ois = sch.SwapConvention(sofr, sofr)
spot = make_calendar('USD').add_business_days(valdate.date(), usd_ois.spot_lag)
swap_rate = swaps.par_rate(swaps.swap_times(sch.make_swap_scheduler(spot, maturity, usd_ois), valdate), rfr_curve)

analytics = {
    'Settlement': f"{settlement:%d-%b-%Y}",
    'Previous coupon': f"{sch.to_date(starts[i]):%d-%b-%Y}",
    'Next coupon': f"{sch.to_date(ends[i]):%d-%b-%Y}",
    'Days accrued': int((np.datetime64(settlement, 'D') - starts[i]).astype(int)),
    'Accrued interest': f"{accrued:.6f}",
    'Clean price': f"{clean_price:.6f}",
    'Dirty price': f"{dirty:.6f}",
    'Yield (to maturity)': f"{ytm:.4%}",
    'Yield (annual)': f"{(1 + ytm / bf.frequency) ** bf.frequency - 1:.4%}",
    'Yield (current)': f"{100 * coupon / clean_price:.4%}",
    'Macaulay duration': f"{bonds.macaulay_duration(bf, coupon, ytm):.4f}",
    'Modified duration': f"{bonds.modified_duration(bf, coupon, ytm):.4f}",
    'DV01 (per 100)': f"{bonds.dv01(bf, coupon, ytm):.6f}",
    'Convexity': f"{bonds.convexity(bf, coupon, ytm):.4f}",
    'Z-spread (bp)': f"{1e4 * zspread:.2f}",
    'Par swap rate to maturity': f"{swap_rate:.4%}",
    'I-spread (bp)': f"{1e4 * (ytm - swap_rate):.2f}",
}
pd.Series(analytics, name='Value').to_frame()

,Value
Settlement,16-Dec-2025
Previous coupon,15-Aug-2025
Next coupon,15-Feb-2026
Days accrued,123
Accrued interest,1.336957
Clean price,98.500000
Dirty price,99.836957
Yield (to maturity),4.1821%
Yield (annual),4.2258%
Yield (current),4.0609%


In [24]:
# Remaining cash-flows at settlement
amounts = bonds.cashflows(bf, coupon)
yield_df = (1.0 + ytm / bf.frequency) ** -bf.periods
curve_df = rfr_curve(bf.pay_times) / rfr_curve(np.array([bf.settle_time]))[0]
z_df = curve_df * np.exp(-zspread * (bf.pay_times - bf.settle_time))

flows_df = pd.DataFrame({'Payment': bf.payment, 'Time': bf.pay_times, 'Accrual': bf.alpha,
                         'Cash flow': amounts, 'DF (yield)': yield_df, 'DF (rfr + Z)': z_df,
                         'PV (yield)': amounts * yield_df, 'PV (rfr + Z)': amounts * z_df})

flows_df.style.format({'Payment': '{:%d-%b-%Y}', 'Accrual': '{:.6f}', 'Time': '{:.6f}',
                       'Cash flow': '{:.4f}', 'DF (yield)': '{:.6f}', 'DF (rfr + Z)': '{:.6f}',
                       'PV (yield)': '{:.6f}', 'PV (rfr + Z)': '{:.6f}'})

,Payment,Time,Accrual,Cash flow,DF (yield),DF (rfr + Z),PV (yield),PV (rfr + Z)
0,17-Feb-2026,0.175342,0.500000,2.0000,0.993163,0.996274,1.986325,1.992547
1,17-Aug-2026,0.671233,0.500000,2.0000,0.972821,0.979585,1.945641,1.959169
2,16-Feb-2027,1.172603,0.500000,2.0000,0.952895,0.959277,1.905790,1.918554
3,16-Aug-2027,1.668493,0.500000,2.0000,0.933378,0.941912,1.866755,1.883824
4,15-Feb-2028,2.169863,0.500000,2.0000,0.914260,0.924305,1.828520,1.848610
5,15-Aug-2028,2.668493,0.500000,2.0000,0.895534,0.906760,1.791068,1.813519
6,15-Feb-2029,3.172603,0.500000,2.0000,0.877192,0.889001,1.754383,1.778002
7,15-Aug-2029,3.668493,0.500000,2.0000,0.859225,0.871525,1.718450,1.743051
8,15-Feb-2030,4.172603,0.500000,2.0000,0.841626,0.853769,1.683252,1.707537
9,15-Aug-2030,4.668493,0.500000,2.0000,0.824388,0.836323,1.648776,1.672645


### Treasury Curve